In [1]:
# ============================================================
# STAGE 1 — DOCUMENT CHARACTERISATION AND QUALITY ASSESSMENT
# D8 — World Bank Project Information Document
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import platform
import re
import subprocess
import sys

import pandas as pd


# antiword is used for the legacy .doc source format.
!apt-get update -qq
!apt-get install -y antiword -qq


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package antiword.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../antiword_0.37-16_amd64.deb ...
Unpacking antiword (0.37-16) ...
Setting up antiword (0.37-16) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D8"

DOCUMENT_NAME = (
    "World Bank — Bhutan - Land Management Project — "
    "Project Information Document (PID), Concept Stage"
)

SOURCE_FORMAT = "DOC"
EXPECTED_PHYSICAL_PAGE_COUNT = 4
EXPECTED_REFERENCE_RECORD_COUNT = 49

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

EXPECTED_CATEGORY_COUNTS = {
    "Project metadata": 13,
    "Development issue": 10,
    "Bank rationale": 2,
    "Project objective": 3,
    "Project component": 3,
    "Safeguard policy": 6,
    "Financing": 7,
    "Contact information": 5
}

MANDATORY_STRING_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Source Location"
]

NULLABLE_STRING_FIELDS = [
    "Unit",
    "Qualifier",
    "Reporting Period"
]

OUTPUT_DIR = Path("outputs_D8_stage1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_VALUES_PATH = (
    OUTPUT_DIR / "D8_reference_values.csv"
)

REFERENCE_VALUES_JSON_PATH = (
    OUTPUT_DIR / "D8_reference_values.json"
)

REFERENCE_SCHEMA_PATH = (
    OUTPUT_DIR / "D8_reference_schema.json"
)

REFERENCE_SUMMARY_PATH = (
    OUTPUT_DIR / "D8_reference_summary.json"
)

DOCUMENT_CHARACTERISATION_PATH = (
    OUTPUT_DIR / "D8_document_characterisation.json"
)

QUALITY_EVIDENCE_PATH = (
    OUTPUT_DIR / "D8_quality_evidence.json"
)

INDICATOR_ASSESSMENT_PATH = (
    OUTPUT_DIR / "D8_indicator_assessment.csv"
)

DIMENSION_ASSESSMENT_PATH = (
    OUTPUT_DIR / "D8_dimension_assessment.csv"
)

REFERENCE_INTEGRITY_PATH = (
    OUTPUT_DIR / "D8_reference_integrity.json"
)

TEXT_DIAGNOSTICS_PATH = (
    OUTPUT_DIR / "D8_text_diagnostics.json"
)

EXTRACTION_SCHEMA_PATH = (
    OUTPUT_DIR / "D8_extraction_schema.json"
)

EXTRACTION_TASK_PATH = (
    OUTPUT_DIR / "D8_extraction_task.txt"
)

REFERENCE_METADATA_PATH = (
    OUTPUT_DIR / "D8_reference_metadata.json"
)

print("Document:", DOCUMENT_ID)
print("Expected physical pages:", EXPECTED_PHYSICAL_PAGE_COUNT)
print("Expected reference records:", EXPECTED_REFERENCE_RECORD_COUNT)
print("Output directory:", OUTPUT_DIR)


Document: D8
Expected physical pages: 4
Expected reference records: 49
Output directory: outputs_D8_stage1


In [3]:
# ============================================================
# 2. Upload source document
# ============================================================

print(
    "Upload the D8 legacy Word document (.doc)."
)

uploaded = files.upload()

doc_paths = [
    Path(filename)
    for filename in uploaded
    if filename.lower().endswith(".doc")
]

if len(doc_paths) != 1:
    raise ValueError(
        "Upload exactly one legacy .doc source document."
    )

SOURCE_PATH = doc_paths[0]

print("Source file:", SOURCE_PATH.name)
print("Source size:", f"{SOURCE_PATH.stat().st_size:,} bytes")


Upload the D8 legacy Word document (.doc).


Saving D8 - Project0Inform1ment010Concept0Stage.doc to D8 - Project0Inform1ment010Concept0Stage.doc
Source file: D8 - Project0Inform1ment010Concept0Stage.doc
Source size: 45,568 bytes


In [4]:
# ============================================================
# 3. File-hashing utility
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

print("Source SHA-256:", SOURCE_SHA256)


Source SHA-256: 61aacfd3138ecfba59fac51d29a970de45d8a909c74b744e756e8a283666c7b5


In [5]:
# ============================================================
# 4. Source-text extraction
# ============================================================

def extract_text_from_legacy_doc(path):
    result = subprocess.run(
        ["antiword", str(path)],
        capture_output=True,
        text=True,
        errors="replace"
    )

    if result.returncode != 0:
        raise RuntimeError(
            "antiword could not extract the D8 source document.\n"
            f"stderr: {result.stderr}"
        )

    text = result.stdout

    if not text.strip():
        raise RuntimeError(
            "The D8 source document produced no extractable text."
        )

    return text


FULL_TEXT = extract_text_from_legacy_doc(
    SOURCE_PATH
)

TEXT_EXTRACTABLE = bool(
    FULL_TEXT.strip()
)

OCR_REQUIRED = False

print("Text extractable:", TEXT_EXTRACTABLE)
print("Extracted characters:", len(FULL_TEXT))
print("Extracted words:", len(FULL_TEXT.split()))


Text extractable: True
Extracted characters: 12317
Extracted words: 1598


In [6]:
# ============================================================
# 5. Source-text diagnostics
# ============================================================

EXPECTED_SECTION_MARKERS = {
    "header":
        "PROJECT INFORMATION DOCUMENT (PID)",
    "concept_stage":
        "CONCEPT STAGE",
    "development_issues":
        "1. Key development issues and rationale for Bank involvement",
    "proposed_objectives":
        "2. Proposed objective(s)",
    "preliminary_description":
        "3. Preliminary description",
    "safeguards":
        "4. Safeguard Policies that Might Apply",
    "financing":
        "5. Tentative financing",
    "contact":
        "6. Contact point"
}

section_marker_status = {
    marker: text.casefold() in FULL_TEXT.casefold()
    for marker, text
    in EXPECTED_SECTION_MARKERS.items()
}

section_markers_valid = all(
    section_marker_status.values()
)


EXPECTED_SOURCE_MARKER_PATTERNS = {
    "AB526":
        r"\bAB526\b",

    "P087039":
        r"\bP087039\b",

    "L-Land degradation":
        r"L-Land\s+degradation",

    "December 4, 2003":
        r"December\s+4,\s+2003",

    "February 4, 2005":
        r"February\s+4,\s+2005",

    "May 27, 2005":
        r"May\s+27,\s+2005",

    "at least 60%":
        r"at\s+least\s+60\s*%",

    "one-quarter":
        r"one-quarter",

    "another 9%":
        r"another\s+9\s*%",

    "reached 520":
        r"reached\s+520",

    "6.7%":
        r"6\.7\s*%",

    "less than 8%":
        r"less\s+than\s+8\s*%",

    "10% of agricultural land":
        r"10\s*%\s+of\s+agricultural\s+land",

    "some 40%":
        r"some\s+40\s*%",

    "One-third of Bhutanese villages":
        r"one-third\s+of\s+Bhutanese\s+villages",

    "up to $0.50 million":
        r"up\s+to\s+\$\s*0\.50\s+million",

    "up to $10.0 million":
        r"up\s+to\s+\$\s*10\.0\s+million",

    "up to $6.0 million":
        r"up\s+to\s+\$\s*6\.0\s+million",

    "OP4.01":
        r"\bOP4\.01\b",

    "OD4.20":
        r"\bOD4\.20\b",

    "OP4.09":
        r"\bOP4\.09\b",

    "OP4.36":
        r"\bOP4\.36\b",

    "Total 16.5":
        r"\|\s*Total\s*\|\s*16\.5\s*\|",

    "Ai Chin Wee":
        r"Ai\s+Chin\s+Wee"
}

source_marker_status = {
    marker: bool(
        re.search(
            pattern,
            FULL_TEXT,
            flags=re.IGNORECASE | re.MULTILINE
        )
    )
    for marker, pattern
    in EXPECTED_SOURCE_MARKER_PATTERNS.items()
}


source_markers_valid = all(
    source_marker_status.values()
)


numeric_tokens = re.findall(
    r"(?<!\w)[£$]?\(?-?\d[\d,]*(?:\.\d+)?%?\)?",
    FULL_TEXT
)

word_count = len(
    FULL_TEXT.split()
)

numeric_token_to_word_ratio = (
    len(numeric_tokens) / word_count
    if word_count
    else 0
)

percentage_tokens = re.findall(
    r"\d+(?:\.\d+)?\s?%",
    FULL_TEXT
)

money_tokens = re.findall(
    r"\$\s?\d+(?:\.\d+)?(?:\s*million)?",
    FULL_TEXT,
    flags=re.IGNORECASE
)

detected_years = sorted(
    set(
        re.findall(
            r"\b(?:19|20)\d{2}\b",
            FULL_TEXT
        )
    )
)


TEXT_DIAGNOSTICS = {
    "document_id": DOCUMENT_ID,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "text_extractable": TEXT_EXTRACTABLE,
    "ocr_required": OCR_REQUIRED,
    "character_count": len(FULL_TEXT),
    "word_count": len(FULL_TEXT.split()),
    "non_empty_line_count": len(
        [
            line
            for line in FULL_TEXT.splitlines()
            if line.strip()
        ]
    ),
    "numeric_token_count": len(numeric_tokens),
    "numeric_token_to_word_ratio":
        round(
            numeric_token_to_word_ratio,
            3
        ),
    "percentage_token_count": len(percentage_tokens),
    "money_token_count": len(money_tokens),
    "detected_years": detected_years,
    "section_marker_status": section_marker_status,
    "section_markers_valid": section_markers_valid,
    "source_marker_status": source_marker_status,
    "source_markers_valid": source_markers_valid,
    "physical_page_count": EXPECTED_PHYSICAL_PAGE_COUNT,
    "page_boundaries_available_in_antiword_text": (
        "\f" in FULL_TEXT
    )
}

print(
    json.dumps(
        TEXT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)

if not section_markers_valid:
    raise AssertionError(
        "One or more required D8 section markers were not found."
    )

if not source_markers_valid:
    raise AssertionError(
        "One or more required D8 source-content markers were not found."
    )


{
  "document_id": "D8",
  "source_file": "D8 - Project0Inform1ment010Concept0Stage.doc",
  "source_file_sha256": "61aacfd3138ecfba59fac51d29a970de45d8a909c74b744e756e8a283666c7b5",
  "text_extractable": true,
  "ocr_required": false,
  "character_count": 12317,
  "word_count": 1598,
  "non_empty_line_count": 183,
  "numeric_token_count": 56,
  "numeric_token_to_word_ratio": 0.035,
  "percentage_token_count": 9,
  "money_token_count": 3,
  "detected_years": [
    "1974",
    "1978",
    "2000",
    "2002",
    "2003",
    "2005"
  ],
  "section_marker_status": {
    "header": true,
    "concept_stage": true,
    "development_issues": true,
    "proposed_objectives": true,
    "preliminary_description": true,
    "safeguards": true,
    "financing": true,
    "contact": true
  },
  "section_markers_valid": true,
  "source_marker_status": {
    "AB526": true,
    "P087039": true,
    "L-Land degradation": true,
    "December 4, 2003": true,
    "February 4, 2005": true,
    "May 27, 2005

In [7]:
# ============================================================
# 6. Document characterisation
# ============================================================

DOCUMENT_CHARACTERISATION = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "source_format": SOURCE_FORMAT,
    "legacy_binary_word_format": True,
    "physical_page_count": EXPECTED_PHYSICAL_PAGE_COUNT,
    "page_boundaries_preserved_by_text_extractor": (
        "\f" in FULL_TEXT
    ),
    "text_extractable": TEXT_EXTRACTABLE,
    "ocr_required": OCR_REQUIRED,
    "total_character_count": len(FULL_TEXT),
    "total_word_count": len(FULL_TEXT.split()),
    "numeric_token_count": len(numeric_tokens),
    "numeric_token_to_word_ratio":
        round(
            numeric_token_to_word_ratio,
            3
        ),
    "percentage_token_count": len(percentage_tokens),
    "money_token_count": len(money_tokens),
    "detected_years": detected_years,
    "represented_sections": [
        "Project metadata header",
        "Key development issues",
        "Rationale for Bank involvement",
        "Proposed objectives",
        "Preliminary description",
        "Safeguard policies that might apply",
        "Tentative financing",
        "Contact point"
    ],
    "fixed_extraction_scope": (
        "Labelled project metadata, fixed quantitative development "
        "observations, two Bank-rationale records, three proposed-objective "
        "records, three project components, six safeguard records, seven "
        "tentative-financing records and five contact records."
    ),
    "excluded_from_reference_scope": [
        "Unlabelled publication-layout artefacts",
        "Narrative examples without a fixed requested record",
        "Comparative statements requiring calculation",
        "Phone numbers embedded only in the implementing-agency header field",
        "Values that would require conversion from one-quarter or one-third",
        "Derived component or financing totals other than the explicitly "
        "represented Total 16.5"
    ],
}

print(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D8",
  "document_name": "World Bank — Bhutan - Land Management Project — Project Information Document (PID), Concept Stage",
  "source_file": "D8 - Project0Inform1ment010Concept0Stage.doc",
  "source_file_sha256": "61aacfd3138ecfba59fac51d29a970de45d8a909c74b744e756e8a283666c7b5",
  "source_format": "DOC",
  "legacy_binary_word_format": true,
  "physical_page_count": 4,
  "page_boundaries_preserved_by_text_extractor": false,
  "text_extractable": true,
  "ocr_required": false,
  "total_character_count": 12317,
  "total_word_count": 1598,
  "numeric_token_count": 56,
  "numeric_token_to_word_ratio": 0.035,
  "percentage_token_count": 9,
  "money_token_count": 3,
  "detected_years": [
    "1974",
    "1978",
    "2000",
    "2002",
    "2003",
    "2005"
  ],
  "represented_sections": [
    "Project metadata header",
    "Key development issues",
    "Rationale for Bank involvement",
    "Proposed objectives",
    "Preliminary description",
    "Safeguard policies tha

In [8]:
# ============================================================
# 7. Fixed extraction task
# ============================================================

EXTRACTION_TASK = """
Extract the fixed project-information records represented in the
supplied World Bank Project Information Document for the
Bhutan - Land Management Project.

Use only the supplied document as evidence.

Return exactly 49 records:

- 13 Project metadata
- 10 Development issue
- 2 Bank rationale
- 3 Project objective
- 3 Project component
- 6 Safeguard policy
- 7 Financing
- 5 Contact information

For every record return exactly these fields:

- Category
- Topic
- Description
- Value
- Unit
- Qualifier
- Reporting Period
- Source Location

Rules:

- Extract only information explicitly represented in the source.
- Use JSON numbers for explicitly represented numeric values.
- Use JSON strings for explicitly represented text, codes and dates.
- Use null where a separate Value, Unit, Qualifier or Reporting Period
  is not explicitly represented.
- Preserve explicit source qualifiers and modifiers such as
  "at least", "less than", "some", "another" and "up to"
  in the Qualifier field.
- Preserve textual quantities such as "one-quarter" and "one-third"
  as textual Values rather than converting them to percentages.
- Preserve source wording, codes and source spellings, including
  represented typographical errors where relevant.
- Do not calculate, infer, derive, convert, repair, harmonise or
  correct source content.
- Return exactly 49 records.
- Return valid JSON using the exact field names defined in the
  extraction schema.
- Do not include explanations before or after the JSON.
"""

print(EXTRACTION_TASK)


Extract the fixed project-information records represented in the
supplied World Bank Project Information Document for the
Bhutan - Land Management Project.

Use only the supplied document as evidence.

Return exactly 49 records:

- 13 Project metadata
- 10 Development issue
- 2 Bank rationale
- 3 Project objective
- 3 Project component
- 6 Safeguard policy
- 7 Financing
- 5 Contact information

For every record return exactly these fields:

- Category
- Topic
- Description
- Value
- Unit
- Qualifier
- Reporting Period
- Source Location

Rules:

- Extract only information explicitly represented in the source.
- Use JSON numbers for explicitly represented numeric values.
- Use JSON strings for explicitly represented text, codes and dates.
- Use null where a separate Value, Unit, Qualifier or Reporting Period
  is not explicitly represented.
- Preserve explicit source qualifiers and modifiers such as
  "at least", "less than", "some", "another" and "up to"
  in the Qualifier field.
- Pre

In [9]:
# ============================================================
# 8. Extraction schema
# ============================================================

EXTRACTION_SCHEMA = {
    "document_id": DOCUMENT_ID,

    "record_level":
        "project_information_record",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Category": {
            "type": ["string", "null"]
        },

        "Topic": {
            "type": ["string", "null"]
        },

        "Description": {
            "type": ["string", "null"]
        },

        "Value": {
            "type": [
                "string",
                "number",
                "null"
            ]
        },

        "Unit": {
            "type": ["string", "null"]
        },

        "Qualifier": {
            "type": ["string", "null"]
        },

        "Reporting Period": {
            "type": ["string", "null"]
        },

        "Source Location": {
            "type": ["string", "null"]
        }
    },

    "expected_output_structure": {
        "document_id": DOCUMENT_ID,

        "records": [
            {
                "Category": "string or null",
                "Topic": "string or null",
                "Description": "string or null",
                "Value": "string, number or null",
                "Unit": "string or null",
                "Qualifier": "string or null",
                "Reporting Period": "string or null",
                "Source Location": "string or null"
            }
        ]
    }
}

print(
    json.dumps(
        EXTRACTION_SCHEMA,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D8",
  "record_level": "project_information_record",
  "expected_record_count": 49,
  "fields": {
    "Category": {
      "type": [
        "string",
        "null"
      ]
    },
    "Topic": {
      "type": [
        "string",
        "null"
      ]
    },
    "Description": {
      "type": [
        "string",
        "null"
      ]
    },
    "Value": {
      "type": [
        "string",
        "number",
        "null"
      ]
    },
    "Unit": {
      "type": [
        "string",
        "null"
      ]
    },
    "Qualifier": {
      "type": [
        "string",
        "null"
      ]
    },
    "Reporting Period": {
      "type": [
        "string",
        "null"
      ]
    },
    "Source Location": {
      "type": [
        "string",
        "null"
      ]
    }
  },
  "expected_output_structure": {
    "document_id": "D8",
    "records": [
      {
        "Category": "string or null",
        "Topic": "string or null",
        "Description": "string or null"

In [10]:
# ============================================================
# 9. Reference schema
# ============================================================

REFERENCE_SCHEMA = {
    "document_id": DOCUMENT_ID,

    "record_level": (
        "One labelled project field, fixed quantitative development "
        "observation, rationale, objective, component, safeguard, "
        "financing source or contact item"
    ),

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "fields": {
        "Category":
            "One of the eight fixed D8 category labels",

        "Topic":
            "The represented project field, issue, objective, "
            "component, policy, financing source or contact item",

        "Description":
            "Source-grounded description of the represented record",

        "Value":
            "Explicit JSON number, source text/code/date, or null",

        "Unit":
            "Measurement unit associated with Value, or null",

        "Qualifier":
            "Explicit source modifier associated with Value, such as "
            "at least, less than, some, another or up to",

        "Reporting Period":
            "Explicitly associated date or period, or null",

        "Source Location":
            "Physical DOC page and source section"
    },

    "null_policy": (
        "Value, Unit, Qualifier and Reporting Period are null when "
        "the corresponding information is not explicitly represented "
        "in the source."
    ),

    "preservation_rules": [
        "Preserve source wording and codes",
        "Preserve source spelling where the document contains a typo",
        "Preserve explicit modifiers in the Qualifier field",
        "Do not convert one-quarter to 25 percent",
        "Do not convert one-third to a percentage",
        "Do not convert component or financing units",
        "Do not infer reporting periods",
        "Do not repair F1, RECEPIENT or Indigenous Pelple"
    ],

    "branch_reuse": (
        "The same fixed reference dataset is reused for "
        "Branches A, B and C."
    )
}

print(
    json.dumps(
        REFERENCE_SCHEMA,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D8",
  "record_level": "One labelled project field, fixed quantitative development observation, rationale, objective, component, safeguard, financing source or contact item",
  "expected_record_count": 49,
  "expected_category_counts": {
    "Project metadata": 13,
    "Development issue": 10,
    "Bank rationale": 2,
    "Project objective": 3,
    "Project component": 3,
    "Safeguard policy": 6,
    "Financing": 7,
    "Contact information": 5
  },
  "fields": {
    "Category": "One of the eight fixed D8 category labels",
    "Topic": "The represented project field, issue, objective, component, policy, financing source or contact item",
    "Description": "Source-grounded description of the represented record",
    "Value": "Explicit JSON number, source text/code/date, or null",
    "Unit": "Measurement unit associated with Value, or null",
    "Qualifier": "Explicit source modifier associated with Value, such as at least, less than, some, another or up to",
   

In [11]:
# ============================================================
# 10. Reference dataset construction
# ============================================================

reference_records = [
    # --------------------------------------------------------
    # Project metadata — DOC page 1
    # --------------------------------------------------------
    {
        "Category": "Project metadata",
        "Topic": "Report No.",
        "Description": "Project Information Document report number",
        "Value": "AB526",
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Header"
    },
    {
        "Category": "Project metadata",
        "Topic": "Project Name",
        "Description": "Project name",
        "Value": "Bhutan - Land Management Project",
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Header"
    },
    {
        "Category": "Project metadata",
        "Topic": "Region",
        "Description": "World Bank region",
        "Value": "SOUTH ASIA",
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Header"
    },
    {
        "Category": "Project metadata",
        "Topic": "Sector",
        "Description": "Represented sector allocation",
        "Value": (
            "General agriculture, fishing and forestry sector (60%);"
            "General public administration sector (20%);"
            "Roads and highways (20%)"
        ),
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Header"
    },
    {
        "Category": "Project metadata",
        "Topic": "Project ID",
        "Description": "World Bank project identifier",
        "Value": "P087039",
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Header"
    },
    {
        "Category": "Project metadata",
        "Topic": "GEF Focal Area",
        "Description": "Global Environment Facility focal area",
        "Value": "L-Land degradation",
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Header"
    },
    {
        "Category": "Project metadata",
        "Topic": "Borrower(s)",
        "Description": "Project borrower",
        "Value": "RGOB",
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Header"
    },
    {
        "Category": "Project metadata",
        "Topic": "Implementing Agency",
        "Description": "Represented implementing-agency status and likely agencies",
        "Value": (
            "TBD (Most likely Ministry of Agriculture and/or Ministry of "
            "Works and Human Settlements), RGOB"
        ),
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Header"
    },
    {
        "Category": "Project metadata",
        "Topic": "Environment Category",
        "Description": "Selected environment category",
        "Value": "B",
        "Unit": "category",
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Header"
    },
    {
        "Category": "Project metadata",
        "Topic": "Safeguard Classification",
        "Description": "Selected safeguard classification",
        "Value": "S2",
        "Unit": "classification",
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Header"
    },
    {
        "Category": "Project metadata",
        "Topic": "Date PID Prepared",
        "Description": "Date PID prepared",
        "Value": "December 4, 2003",
        "Unit": "date",
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Header"
    },
    {
        "Category": "Project metadata",
        "Topic": "Estimated Date of Appraisal Authorization",
        "Description": "Estimated appraisal-authorization date",
        "Value": "February 4, 2005",
        "Unit": "date",
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Header"
    },
    {
        "Category": "Project metadata",
        "Topic": "Estimated Date of Board Approval",
        "Description": "Estimated Board-approval date",
        "Value": "May 27, 2005",
        "Unit": "date",
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Header"
    },

    # --------------------------------------------------------
    # Development issues — DOC pages 1–2
    # --------------------------------------------------------
    {
        "Category": "Development issue",
        "Topic": "Forest policy",
        "Description": (
            "Share of Bhutanese territory required to remain forested "
            "in perpetuity"
        ),
        "Value": 60,
        "Unit": "percent",
        "Qualifier": "at least",
        "Reporting Period": "Since 1974",
        "Source Location": "DOC page 1 — Section 1, Key Development Issues"
    },
    {
        "Category": "Development issue",
        "Topic": "Protected areas",
        "Description": "Share of the country's area set aside for protected areas",
        "Value": "one-quarter",
        "Unit": "of country area",
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Section 1, Key Development Issues"
    },
    {
        "Category": "Development issue",
        "Topic": "Wildlife corridors",
        "Description": "Additional area offered for wildlife corridors",
        "Value": 9,
        "Unit": "percent",
        "Qualifier": "another",
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Section 1, Key Development Issues"
    },
    {
        "Category": "Development issue",
        "Topic": "Population density",
        "Description": "Population density per square kilometre of arable land",
        "Value": 520,
        "Unit": "people per sq. km of arable land",
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Section 1, Key Development Issues"
    },
    {
        "Category": "Development issue",
        "Topic": "Urban growth rate",
        "Description": "Bhutan urban growth rate",
        "Value": 6.7,
        "Unit": "percent",
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Section 1, Key Development Issues"
    },
    {
        "Category": "Development issue",
        "Topic": "Arable land",
        "Description": "Arable land as a share of land area",
        "Value": 8,
        "Unit": "percent",
        "Qualifier": "less than",
        "Reporting Period": None,
        "Source Location": "DOC page 1 — Section 1, Key Development Issues"
    },
    {
        "Category": "Development issue",
        "Topic": "Agricultural land affected by water erosion",
        "Description": "Agricultural land affected by water erosion",
        "Value": 10,
        "Unit": "percent",
        "Reporting Period": None,
        "Source Location": "DOC page 2 — Section 1, Key Development Issues"
    },
    {
        "Category": "Development issue",
        "Topic": "Hydropower revenue contribution",
        "Description": (
            "Development budget underwritten by hydropower-industry revenue"
        ),
        "Value": 40,
        "Unit": "percent",
        "Qualifier": "some",
        "Reporting Period": None,
        "Source Location": "DOC page 2 — Section 1, Key Development Issues"
    },
    {
        "Category": "Development issue",
        "Topic": "Villages without feeder roads",
        "Description": "Bhutanese villages not connected to feeder roads",
        "Value": "one-third",
        "Unit": "of Bhutanese villages",
        "Reporting Period": "2000 Poverty Assessment and Analysis Report",
        "Source Location": "DOC page 2 — Section 1, Key Development Issues"
    },
    {
        "Category": "Development issue",
        "Topic": "Villages facing food insecurity",
        "Description": "Bhutanese villages facing food insecurity",
        "Value": "one-third",
        "Unit": "of Bhutanese villages",
        "Reporting Period": "2000 Poverty Assessment and Analysis Report",
        "Source Location": "DOC page 2 — Section 1, Key Development Issues"
    },

    # --------------------------------------------------------
    # Bank rationale — DOC page 2
    # --------------------------------------------------------
    {
        "Category": "Bank rationale",
        "Topic": "Cross-sectoral accountability and incentives",
        "Description": (
            "The current sector-oriented institutional framework is not "
            "able to provide effective cross-sectoral accountability and "
            "incentive mechanisms that can mitigate landscape-degradation "
            "pressures"
        ),
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 2 — Section 1, Rationale for Bank Involvement"
    },
    {
        "Category": "Bank rationale",
        "Topic": "Candidate for GEF support",
        "Description": (
            "Bhutan's governance, political will and early adoption of "
            "environmental-stewardship approaches are presented as positive "
            "factors making it a good candidate for GEF support in sustainable "
            "land management"
        ),
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 2 — Section 1, Rationale for Bank Involvement"
    },

    # --------------------------------------------------------
    # Proposed objectives — DOC page 3
    # --------------------------------------------------------
    {
        "Category": "Project objective",
        "Topic": "Sustainable land management practices",
        "Description": (
            "Promote innovative technical and institutional mechanisms to "
            "enhance sustainable land management practices with local, "
            "regional and global environmental benefits"
        ),
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 3 — Section 2, Proposed objective(s)"
    },
    {
        "Category": "Project objective",
        "Topic": "Technical innovations and ecosystem functions",
        "Description": (
            "Test and demonstrate technical innovations that reduce land "
            "degradation and related downstream impacts, focus land-use "
            "planning on ecosystem functions and services, and develop "
            "cross-sectoral monitoring and management mechanisms"
        ),
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 3 — Section 2, Proposed objective(s)"
    },
    {
        "Category": "Project objective",
        "Topic": "Multi-sectoral planning and local participation",
        "Description": (
            "Address land and watershed degradation through multi-sectoral "
            "planning and implementation with strong local participation"
        ),
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 3 — Section 2, Proposed objective(s)"
    },

    # --------------------------------------------------------
    # Project components — DOC page 3
    # --------------------------------------------------------
    {
        "Category": "Project component",
        "Topic": "Component 1",
        "Description": (
            "An improved policy and planning framework for sustainable "
            "land management"
        ),
        "Value": 0.50,
        "Unit": "USD million",
        "Qualifier": "up to",
        "Reporting Period": None,
        "Source Location": "DOC page 3 — Section 3, Preliminary description"
    },
    {
        "Category": "Project component",
        "Topic": "Component 2",
        "Description": "Integrated land management sub-projects",
        "Value": 10.0,
        "Unit": "USD million",
        "Qualifier": "up to",
        "Reporting Period": None,
        "Source Location": "DOC page 3 — Section 3, Preliminary description"
    },
    {
        "Category": "Project component",
        "Topic": "Component 3",
        "Description": "Institution-strengthening and capacity-building",
        "Value": 6.0,
        "Unit": "USD million",
        "Qualifier": "up to",
        "Reporting Period": None,
        "Source Location": "DOC page 3 — Section 3, Preliminary description"
    },

    # --------------------------------------------------------
    # Safeguards — DOC page 4
    # --------------------------------------------------------
    {
        "Category": "Safeguard policy",
        "Topic": "Environmental Assessment",
        "Description": "Potentially applicable safeguard policy",
        "Value": "OP4.01",
        "Unit": "policy code",
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 4, Safeguard Policies that Might Apply"
    },
    {
        "Category": "Safeguard policy",
        "Topic": "Indigenous Pelple",
        "Description": "Potentially applicable safeguard policy",
        "Value": "OD4.20",
        "Unit": "policy code",
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 4, Safeguard Policies that Might Apply"
    },
    {
        "Category": "Safeguard policy",
        "Topic": "Natural Habitats",
        "Description": "Potentially applicable safeguard policy",
        "Value": "OP4.09",
        "Unit": "policy code",
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 4, Safeguard Policies that Might Apply"
    },
    {
        "Category": "Safeguard policy",
        "Topic": "Forests",
        "Description": "Potentially applicable safeguard policy",
        "Value": "OP4.36",
        "Unit": "policy code",
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 4, Safeguard Policies that Might Apply"
    },
    {
        "Category": "Safeguard policy",
        "Topic": "Environmental assessment category",
        "Description": "Environmental assessment category now assessed",
        "Value": "B",
        "Unit": "category",
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 4, Safeguard Policies that Might Apply"
    },
    {
        "Category": "Safeguard policy",
        "Topic": "Potential community-grant category",
        "Description": (
            "EA category could be F1 if the project design includes grants "
            "to communities for sub-projects with environmental implications"
        ),
        "Value": "F1",
        "Unit": "category",
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 4, Safeguard Policies that Might Apply"
    },

    # --------------------------------------------------------
    # Tentative financing — DOC page 4
    # --------------------------------------------------------
    {
        "Category": "Financing",
        "Topic": "BORROWER/RECEPIENT",
        "Description": "Tentative financing contribution",
        "Value": 0.5,
        "Unit": "USD million",
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 5, Tentative financing"
    },
    {
        "Category": "Financing",
        "Topic": "GLOBAL ENVIRONMENT FACILITY",
        "Description": "Tentative financing contribution",
        "Value": 7.5,
        "Unit": "USD million",
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 5, Tentative financing"
    },
    {
        "Category": "Financing",
        "Topic": "LOCAL COMMUNITIES",
        "Description": "Tentative financing contribution",
        "Value": 1.5,
        "Unit": "USD million",
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 5, Tentative financing"
    },
    {
        "Category": "Financing",
        "Topic": (
            "LOCAL GOVTS. (PROV., DISTRICT, CITY) OF BORROWING COUNTRY"
        ),
        "Description": "Tentative financing contribution",
        "Value": 1,
        "Unit": "USD million",
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 5, Tentative financing"
    },
    {
        "Category": "Financing",
        "Topic": "BILATERAL AGENCIES (UNIDENTIFIED)",
        "Description": "Tentative financing contribution",
        "Value": 5,
        "Unit": "USD million",
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 5, Tentative financing"
    },
    {
        "Category": "Financing",
        "Topic": (
            "NON-GOVERNMENT ORGANIZATION (NGO) OF BORROWING COUNTRY"
        ),
        "Description": "Tentative financing contribution",
        "Value": 1,
        "Unit": "USD million",
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 5, Tentative financing"
    },
    {
        "Category": "Financing",
        "Topic": "Total",
        "Description": "Total tentative financing",
        "Value": 16.5,
        "Unit": "USD million",
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 5, Tentative financing"
    },

    # --------------------------------------------------------
    # Contact point — DOC page 4
    # --------------------------------------------------------
    {
        "Category": "Contact information",
        "Topic": "Contact",
        "Description": "World Bank contact person",
        "Value": "Ai Chin Wee",
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 6, Contact point"
    },
    {
        "Category": "Contact information",
        "Topic": "Title",
        "Description": "World Bank contact title",
        "Value": "Sr Operations Off.",
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 6, Contact point"
    },
    {
        "Category": "Contact information",
        "Topic": "Tel",
        "Description": "World Bank contact telephone",
        "Value": "(202) 458-5049",
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 6, Contact point"
    },
    {
        "Category": "Contact information",
        "Topic": "Fax",
        "Description": "World Bank contact fax",
        "Value": "(202) 614-1049",
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 6, Contact point"
    },
    {
        "Category": "Contact information",
        "Topic": "Email",
        "Description": "World Bank contact email",
        "Value": "Awee@worldbank.org",
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "DOC page 4 — Section 6, Contact point"
    }
]


reference_values_df = pd.DataFrame(
    reference_records,
    columns=EXPECTED_FIELDS
)

reference_values_df = (
    reference_values_df
    .astype(object)
    .where(
        pd.notna(reference_values_df),
        None
    )
)

reference_records = (
    reference_values_df
    .to_dict(
        orient="records"
    )
)

print("Reference records:", len(reference_values_df))

display(reference_values_df.head(15))


Reference records: 49


,Category,Topic,Description,Value,Unit,Qualifier,Reporting Period,Source Location
0,Project metadata,Report No.,Project Information Document report number,AB526,None,None,None,DOC page 1 — Header
1,Project metadata,Project Name,Project name,Bhutan - Land Management Project,None,None,None,DOC page 1 — Header
2,Project metadata,Region,World Bank region,SOUTH ASIA,None,None,None,DOC page 1 — Header
3,Project metadata,Sector,Represented sector allocation,"General agriculture, fishing and forestry sect...",None,None,None,DOC page 1 — Header
4,Project metadata,Project ID,World Bank project identifier,P087039,None,None,None,DOC page 1 — Header
5,Project metadata,GEF Focal Area,Global Environment Facility focal area,L-Land degradation,None,None,None,DOC page 1 — Header
6,Project metadata,Borrower(s),Project borrower,RGOB,None,None,None,DOC page 1 — Header
7,Project metadata,Implementing Agency,Represented implementing-agency status and lik...,TBD (Most likely Ministry of Agriculture and/o...,None,None,None,DOC page 1 — Header
8,Project metadata,Environment Category,Selected environment category,B,category,None,None,DOC page 1 — Header
9,Project metadata,Safeguard Classification,Selected safeguard classification,S2,classification,None,None,DOC page 1 — Header


In [12]:
# ============================================================
# 11. Reference schema validation
# ============================================================

reference_schema_valid = (
    reference_values_df.columns.tolist()
    == EXPECTED_FIELDS
)

record_count_valid = (
    len(reference_values_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)

observed_category_counts = (
    reference_values_df["Category"]
    .value_counts()
    .to_dict()
)

category_counts_valid = (
    observed_category_counts
    == EXPECTED_CATEGORY_COUNTS
)


type_issue_rows = []
missing_mandatory_rows = []

for row_index, row in reference_values_df.iterrows():

    for field in MANDATORY_STRING_FIELDS:
        value = row[field]

        if value is None or value == "":
            missing_mandatory_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field
                }
            )

        elif not isinstance(value, str):
            type_issue_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field,
                    "Observed Type": type(value).__name__
                }
            )

    for field in NULLABLE_STRING_FIELDS:
        value = row[field]

        if (
            value is not None
            and not isinstance(value, str)
        ):
            type_issue_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field,
                    "Observed Type": type(value).__name__
                }
            )

    value = row["Value"]

    if (
        value is not None
        and (
            isinstance(value, bool)
            or not isinstance(
                value,
                (str, int, float)
            )
        )
    ):
        type_issue_rows.append(
            {
                "Record Index": int(row_index),
                "Field": "Value",
                "Observed Type": type(value).__name__
            }
        )


type_issues_df = pd.DataFrame(
    type_issue_rows
)

missing_mandatory_df = pd.DataFrame(
    missing_mandatory_rows
)

field_types_valid = (
    type_issues_df.empty
)

mandatory_fields_complete = (
    missing_mandatory_df.empty
)


print("Reference schema valid:", reference_schema_valid)
print("Record count valid:", record_count_valid)
print("Category counts valid:", category_counts_valid)
print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)

print(
    json.dumps(
        observed_category_counts,
        ensure_ascii=False,
        indent=2
    )
)


if not reference_schema_valid:
    raise AssertionError(
        "D8 reference schema is invalid."
    )

if not record_count_valid:
    raise AssertionError(
        f"Expected {EXPECTED_REFERENCE_RECORD_COUNT} records, "
        f"found {len(reference_values_df)}."
    )

if not category_counts_valid:
    raise AssertionError(
        "D8 reference category counts are invalid."
    )

if not field_types_valid:
    display(type_issues_df)

    raise AssertionError(
        "D8 reference field types are invalid."
    )

if not mandatory_fields_complete:
    display(missing_mandatory_df)

    raise AssertionError(
        "D8 mandatory reference fields are incomplete."
    )


Reference schema valid: True
Record count valid: True
Category counts valid: True
Field types valid: True
Mandatory fields complete: True
{
  "Project metadata": 13,
  "Development issue": 10,
  "Financing": 7,
  "Safeguard policy": 6,
  "Contact information": 5,
  "Project objective": 3,
  "Project component": 3,
  "Bank rationale": 2
}


In [13]:
# ============================================================
# 12. Reference integrity checks
# ============================================================

duplicate_mask = (
    reference_values_df
    .duplicated(
        subset=EXPECTED_FIELDS,
        keep=False
    )
)

duplicate_record_count = int(
    duplicate_mask.sum()
)

source_location_pattern_valid = bool(
    reference_values_df[
        "Source Location"
    ].map(
        lambda value: bool(
            re.match(
                r"^DOC page [1-4] — .+$",
                value
            )
        )
    ).all()
)

source_page_numbers = (
    reference_values_df[
        "Source Location"
    ]
    .str.extract(
        r"^DOC page (\d+)"
    )[0]
    .astype(int)
)

source_pages_within_document = bool(
    source_page_numbers.between(
        1,
        EXPECTED_PHYSICAL_PAGE_COUNT
    ).all()
)


expected_qualifiers = {
    "at least",
    "another",
    "less than",
    "some",
    "up to"
}

observed_qualifiers = set(
    reference_values_df[
        "Qualifier"
    ].dropna()
)

qualifier_values_valid = (
    observed_qualifiers
    == expected_qualifiers
)


qualifier_embedded_in_unit_count = int(
    reference_values_df[
        "Unit"
    ]
    .fillna("")
    .str.contains(
        r"\b(at least|another|less than|some|up to)\b",
        case=False,
        regex=True
    )
    .sum()
)


textual_quantity_values = {
    "one-quarter",
    "one-third"
}

observed_textual_quantities = set(
    reference_values_df.loc[
        reference_values_df[
            "Value"
        ].isin(
            textual_quantity_values
        ),
        "Value"
    ]
)

textual_quantities_preserved = (
    observed_textual_quantities
    == textual_quantity_values
)


print(
    "Observed qualifiers:",
    sorted(observed_qualifiers)
)

print(
    "Qualifier values valid:",
    qualifier_values_valid
)

print(
    "Qualifiers embedded in Unit:",
    qualifier_embedded_in_unit_count
)

print(
    "Textual quantities preserved:",
    textual_quantities_preserved
)


if not qualifier_values_valid:
    raise AssertionError(
        "Unexpected or missing D8 qualifier values."
    )

if qualifier_embedded_in_unit_count != 0:
    raise AssertionError(
        "One or more D8 qualifiers remain embedded "
        "inside the Unit field."
    )

if not textual_quantities_preserved:
    raise AssertionError(
        "one-quarter or one-third was not preserved "
        "as a source-grounded textual value."
    )


source_typo_markers = [
    "Indigenous Pelple",
    "F1",
    "BORROWER/RECEPIENT"
]

source_typos_preserved = all(
    marker in (
        reference_values_df.astype(str)
        .agg(" ".join, axis=1)
        .str.cat(sep=" ")
    )
    for marker in source_typo_markers
)


explicit_numeric_record_count = int(
    reference_values_df[
        "Value"
    ].map(
        lambda value: (
            isinstance(value, (int, float))
            and not isinstance(value, bool)
        )
    ).sum()
)

text_value_record_count = int(
    reference_values_df[
        "Value"
    ].map(
        lambda value: isinstance(value, str)
    ).sum()
)

null_value_record_count = int(
    reference_values_df[
        "Value"
    ].isna().sum()
)

print("Duplicate records:", duplicate_record_count)
print("Source-location format valid:", source_location_pattern_valid)
print("Source pages within document:", source_pages_within_document)

print(
    "Qualifier values valid:",
    qualifier_values_valid
)

print(
    "Qualifiers embedded in Unit:",
    qualifier_embedded_in_unit_count
)

print(
    "Textual quantities preserved:",
    textual_quantities_preserved
)

print("Source typos preserved:", source_typos_preserved)
print("Numeric-value records:", explicit_numeric_record_count)
print("Text-value records:", text_value_record_count)
print("Null-value records:", null_value_record_count)

if duplicate_record_count != 0:
    display(
        reference_values_df.loc[
            duplicate_mask
        ]
    )

    raise AssertionError(
        "Unexpected duplicate D8 reference records detected."
    )

if not source_location_pattern_valid:
    raise AssertionError(
        "One or more D8 source locations have an invalid format."
    )

if not source_pages_within_document:
    raise AssertionError(
        "A D8 source location refers to a page outside the document."
    )

if not source_typos_preserved:
    raise AssertionError(
        "One or more required represented source spellings were repaired."
    )


Observed qualifiers: ['another', 'at least', 'less than', 'some', 'up to']
Qualifier values valid: True
Qualifiers embedded in Unit: 0
Textual quantities preserved: True
Duplicate records: 0
Source-location format valid: True
Source pages within document: True
Qualifier values valid: True
Qualifiers embedded in Unit: 0
Textual quantities preserved: True
Source typos preserved: True
Numeric-value records: 17
Text-value records: 27
Null-value records: 5


/tmp/ipykernel_6027/500167615.py:73: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


In [14]:
# ============================================================
# 13. Reference summary and integrity status
# ============================================================

REFERENCE_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "reference_record_count":
        int(len(reference_values_df)),
    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,
    "record_count_valid":
        record_count_valid,
    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts":
        observed_category_counts,
    "category_counts_valid":
        category_counts_valid,
    "numeric_value_record_count":
        explicit_numeric_record_count,
    "text_value_record_count":
        text_value_record_count,
    "null_value_record_count":
        null_value_record_count,
    "source_pages_represented":
        sorted(
            source_page_numbers.unique().tolist()
        ),
    "fields":
        EXPECTED_FIELDS
}


REFERENCE_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "source_format": SOURCE_FORMAT,
    "physical_page_count":
        EXPECTED_PHYSICAL_PAGE_COUNT,
    "text_extractable":
        TEXT_EXTRACTABLE,
    "ocr_required":
        OCR_REQUIRED,
    "section_markers_valid":
        section_markers_valid,
    "source_markers_valid":
        source_markers_valid,
    "reference_schema_valid":
        reference_schema_valid,
    "record_count_valid":
        record_count_valid,
    "category_counts_valid":
        category_counts_valid,
    "field_types_valid":
        field_types_valid,
    "mandatory_fields_complete":
        mandatory_fields_complete,
    "duplicate_record_count":
        duplicate_record_count,
    "source_location_pattern_valid":
        source_location_pattern_valid,
    "source_pages_within_document":
        source_pages_within_document,
    "qualifier_values_valid":
        qualifier_values_valid,
    "qualifier_embedded_in_unit_count":
        qualifier_embedded_in_unit_count,
    "textual_quantities_preserved":
        textual_quantities_preserved,
    "source_typos_preserved":
        source_typos_preserved,
    "manual_reference_construction":
        True,
    "calculation_applied":
        False,
    "inference_applied":
        False,
    "unit_conversion_applied":
        False,
    "source_value_repair_applied":
        False,
    "reference_integrity_passed": all(
        [
            TEXT_EXTRACTABLE,
            section_markers_valid,
            source_markers_valid,
            reference_schema_valid,
            record_count_valid,
            category_counts_valid,
            field_types_valid,
            mandatory_fields_complete,
            duplicate_record_count == 0,
            source_location_pattern_valid,
            source_pages_within_document,
            qualifier_values_valid,
            qualifier_embedded_in_unit_count == 0,
            textual_quantities_preserved,
            source_typos_preserved
        ]
    )
}


print(
    json.dumps(
        REFERENCE_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)

print(
    json.dumps(
        REFERENCE_INTEGRITY,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D8",
  "document_name": "World Bank — Bhutan - Land Management Project — Project Information Document (PID), Concept Stage",
  "reference_record_count": 49,
  "expected_record_count": 49,
  "record_count_valid": true,
  "expected_category_counts": {
    "Project metadata": 13,
    "Development issue": 10,
    "Bank rationale": 2,
    "Project objective": 3,
    "Project component": 3,
    "Safeguard policy": 6,
    "Financing": 7,
    "Contact information": 5
  },
  "observed_category_counts": {
    "Project metadata": 13,
    "Development issue": 10,
    "Financing": 7,
    "Safeguard policy": 6,
    "Contact information": 5,
    "Project objective": 3,
    "Project component": 3,
    "Bank rationale": 2
  },
  "category_counts_valid": true,
  "numeric_value_record_count": 17,
  "text_value_record_count": 27,
  "null_value_record_count": 5,
  "source_pages_represented": [
    1,
    2,
    3,
    4
  ],
  "fields": [
    "Category",
    "Topic",
    "Description",


In [15]:
# ============================================================
# 14. Reference metadata
# ============================================================

REFERENCE_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "source_format":
        SOURCE_FORMAT,

    "reference_file":
        "D8_reference_values.csv",

    "reference_construction_method":
        "Manual document-grounded construction",

    "expected_reference_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_record_count":
        int(
            len(reference_values_df)
        ),

    "reference_fields":
        EXPECTED_FIELDS,

    "source_locations_manually_verified":
        True,

    "page_locations_derived_from_antiword":
        False,

    "antiword_used_for_text_diagnostics":
        True,

    "manual_calculation_applied":
        False,

    "semantic_inference_applied":
        False,

    "unit_conversion_applied":
        False,

    "source_value_repair_applied":
        False,

    "reference_values_branch_independent":
        True,

    "reference_values_to_be_reused_for_branches": [
        "A",
        "B",
        "C"
    ],

    "notes": (
        "The source is a legacy binary Word document. antiword is used "
        "for machine-readable text diagnostics, but physical page "
        "locations in the fixed reference dataset were manually "
        "verified against the original four-page source."
    )
}

print(
    json.dumps(
        REFERENCE_METADATA,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D8",
  "document_name": "World Bank — Bhutan - Land Management Project — Project Information Document (PID), Concept Stage",
  "source_file": "D8 - Project0Inform1ment010Concept0Stage.doc",
  "source_file_sha256": "61aacfd3138ecfba59fac51d29a970de45d8a909c74b744e756e8a283666c7b5",
  "source_format": "DOC",
  "reference_file": "D8_reference_values.csv",
  "reference_construction_method": "Manual document-grounded construction",
  "expected_reference_record_count": 49,
  "observed_reference_record_count": 49,
  "reference_fields": [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
  ],
  "source_locations_manually_verified": true,
  "page_locations_derived_from_antiword": false,
  "antiword_used_for_text_diagnostics": true,
  "manual_calculation_applied": false,
  "semantic_inference_applied": false,
  "unit_conversion_applied": false,
  "source_value_repair_applied": false,
  "refe

In [16]:
# ============================================================
# 15. Indicator-level document assessment
# ============================================================

indicator_assessment = [
    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Reading Order Quality",

        "Score":
            "Medium",

        "Evidence Source":
            "antiword text extraction + manual document inspection",

        "Justification":
            "The document has a generally coherent section sequence, "
            "but it combines labelled metadata, narrative text, "
            "component descriptions, safeguard lists, financing "
            "information and contact fields, while the legacy DOC "
            "text extraction does not preserve physical page boundaries."
    },

    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Table Structure Integrity",

        "Score":
            "Medium",

        "Evidence Source":
            "Manual source inspection + extracted-text inspection",

        "Justification":
            "Project metadata and financing information follow "
            "structured layouts, but table-like relationships and "
            "wrapped rows are represented through legacy Word layout "
            "rather than explicit machine-readable table structures."
    },

    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Section/Header Hierarchy",

        "Score":
            "Low",

        "Evidence Source":
            "Section-marker detection + manual inspection",

        "Justification":
            "The document contains explicit numbered section headings "
            "and clearly labelled project-information fields, making "
            "the overall hierarchy readily identifiable."
    },

    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "Sharpness",

        "Score":
            "Low",

        "Evidence Source":
            "Manual source inspection",

        "Justification":
            "The source content is machine readable and no image "
            "quality limitation constrains text recovery."
    },

    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "Noise / Degradation",

        "Score":
            "Low",

        "Evidence Source":
            "Manual source inspection",

        "Justification":
            "No scanning noise, blur or visual degradation affects "
            "the recoverability of the document content."
    },

    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "OCR Dependency",

        "Score":
            "Low",

        "Evidence Source":
            "Automated antiword text extraction",

        "Justification":
            "The legacy Word document contains machine-readable text "
            "and does not require OCR."
    },

    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Terminology Consistency",

        "Score":
            "Medium",

        "Evidence Source":
            "Manual content inspection",

        "Justification":
            "Project-development and environmental terminology is "
            "generally coherent, but the document combines project "
            "administration, environmental policy, safeguards, "
            "financing and contact terminology."
    },

    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Schema Alignment",

        "Score":
            "High",

        "Evidence Source":
            "Reference-schema comparison",

        "Justification":
            "The common extraction schema must represent heterogeneous "
            "record types including metadata fields, narrative "
            "development observations, objectives, components, "
            "safeguards, financing sources and contact information."
    },

    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Numerical Density",

        "Score":
            "Medium",

        "Evidence Source":
            "Automated text profiling + manual inspection",

        "Justification":
            "The document contains multiple percentages, dates, cost "
            "estimates and financing amounts, but narrative project "
            "description remains a substantial proportion of the source."
    },

    {
        "Dimension":
            "Completeness and Consistency",

        "Indicator":
            "Required Field Presence",

        "Score":
            "Low",

        "Evidence Source":
            "Reference-value verification",

        "Justification":
            "All information required for the predefined 49-record "
            "extraction scope is represented in the supplied document."
    },

    {
        "Dimension":
            "Completeness and Consistency",

        "Indicator":
            "Internal Consistency",

        "Score":
            "Medium",

        "Evidence Source":
            "Manual source and reference inspection",

        "Justification":
            "The document is internally coherent, but source "
            "typographical errors, checkbox codes and differing "
            "representation conventions require careful preservation "
            "during comparison."
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",

        "Indicator":
            "Format Heterogeneity",

        "Score":
            "High",

        "Evidence Source":
            "Legacy-format diagnostics + manual inspection",

        "Justification":
            "The source is a legacy binary DOC and combines labelled "
            "metadata, narrative sections, checkbox notation, "
            "component descriptions, safeguard lists, financing rows "
            "and contact information."
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",

        "Indicator":
            "Unit / Label Variability",

        "Score":
            "High",

        "Evidence Source":
            "Reference and source inspection",

        "Justification":
            "The extraction scope contains percentages, textual "
            "fractions, population density, USD millions, categorical "
            "codes, dates and explicit modifiers such as at least, "
            "less than, some, another and up to."
    }
]


indicator_assessment_df = pd.DataFrame(
    indicator_assessment
)

display(
    indicator_assessment_df
)

,Dimension,Indicator,Score,Evidence Source,Justification
0,Structural Readiness,Reading Order Quality,Medium,antiword text extraction + manual document ins...,The document has a generally coherent section ...
1,Structural Readiness,Table Structure Integrity,Medium,Manual source inspection + extracted-text insp...,Project metadata and financing information fol...
2,Structural Readiness,Section/Header Hierarchy,Low,Section-marker detection + manual inspection,The document contains explicit numbered sectio...
3,Visual/OCR Readiness,Sharpness,Low,Manual source inspection,The source content is machine readable and no ...
4,Visual/OCR Readiness,Noise / Degradation,Low,Manual source inspection,"No scanning noise, blur or visual degradation ..."
5,Visual/OCR Readiness,OCR Dependency,Low,Automated antiword text extraction,The legacy Word document contains machine-read...
6,Semantic Quality,Terminology Consistency,Medium,Manual content inspection,Project-development and environmental terminol...
7,Semantic Quality,Schema Alignment,High,Reference-schema comparison,The common extraction schema must represent he...
8,Semantic Quality,Numerical Density,Medium,Automated text profiling + manual inspection,"The document contains multiple percentages, da..."
9,Completeness and Consistency,Required Field Presence,Low,Reference-value verification,All information required for the predefined 49...


In [17]:
# ============================================================
# 16. Validate indicator assessment
# ============================================================

VALID_SCORES = {
    "Low",
    "Medium",
    "High"
}

expected_indicators = {
    "Reading Order Quality",
    "Table Structure Integrity",
    "Section/Header Hierarchy",
    "Sharpness",
    "Noise / Degradation",
    "OCR Dependency",
    "Terminology Consistency",
    "Schema Alignment",
    "Numerical Density",
    "Required Field Presence",
    "Internal Consistency",
    "Format Heterogeneity",
    "Unit / Label Variability"
}

invalid_scores = (
    set(
        indicator_assessment_df[
            "Score"
        ].dropna().unique()
    )
    - VALID_SCORES
)

observed_indicators = set(
    indicator_assessment_df[
        "Indicator"
    ]
)

missing_indicators = (
    expected_indicators
    - observed_indicators
)

unexpected_indicators = (
    observed_indicators
    - expected_indicators
)

if invalid_scores:
    raise ValueError(
        f"Invalid indicator scores: {invalid_scores}"
    )

if missing_indicators:
    raise ValueError(
        f"Missing indicators: {missing_indicators}"
    )

if unexpected_indicators:
    raise ValueError(
        f"Unexpected indicators: {unexpected_indicators}"
    )

if len(indicator_assessment_df) != len(
    expected_indicators
):
    raise ValueError(
        "Duplicate indicator rows detected."
    )

print(
    "Indicator assessment validation passed."
)

Indicator assessment validation passed.


In [18]:
# ============================================================
# 17. Dimension-level assessment
# ============================================================

score_to_numeric = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

indicator_assessment_df[
    "Numeric Score"
] = indicator_assessment_df[
    "Score"
].map(
    score_to_numeric
)


dimension_assessment_df = (
    indicator_assessment_df
    .groupby(
        "Dimension",
        as_index=False
    )
    .agg(
        Mean_Score=(
            "Numeric Score",
            "mean"
        ),
        Number_of_Indicators=(
            "Indicator",
            "count"
        )
    )
)


def classify_dimension_score(
    mean_score
):
    if mean_score < 1.5:
        return "Low"

    elif mean_score < 2.5:
        return "Medium"

    else:
        return "High"


dimension_assessment_df[
    "Dimension Score"
] = dimension_assessment_df[
    "Mean_Score"
].apply(
    classify_dimension_score
)

dimension_assessment_df[
    "Mean_Score"
] = dimension_assessment_df[
    "Mean_Score"
].round(2)

display(
    dimension_assessment_df
)

,Dimension,Mean_Score,Number_of_Indicators,Dimension Score
0,Completeness and Consistency,1.50,2,Medium
1,Representation and Normalisation Complexity,3.00,2,High
2,Semantic Quality,2.33,3,Medium
3,Structural Readiness,1.67,3,Medium
4,Visual/OCR Readiness,1.00,3,Low


In [19]:
# ============================================================
# 18. Structured quality-assessment evidence
# ============================================================

QUALITY_EVIDENCE = {
    "document_id":
        DOCUMENT_ID,

    "assessment_basis":
        "Observed document evidence was mapped to the "
        "predefined Low, Medium, and High operational "
        "criteria defined in Table 3.3 of the methodology.",

    "evidence_method":
        "Evidence was obtained through automated profiling "
        "where measurable characteristics could be derived "
        "programmatically and through documented manual "
        "inspection where qualitative assessment was required.",

    "indicators":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_aggregation": {
        "encoding": {
            "Low": 1,
            "Medium": 2,
            "High": 3
        },

        "aggregation":
            "Arithmetic mean of indicator scores within "
            "each dimension.",

        "classification_rule": {
            "Low":
                "mean < 1.5",

            "Medium":
                "1.5 <= mean < 2.5",

            "High":
                "mean >= 2.5"
        }
    },

    "dimensions":
        dimension_assessment_df[
            [
                "Dimension",
                "Mean_Score",
                "Dimension Score"
            ]
        ].to_dict(
            orient="records"
        )
}

In [20]:
# ============================================================
# 19. Export Stage 1 outputs
# ============================================================

reference_values_df.to_csv(
    REFERENCE_VALUES_PATH,
    index=False,
    encoding="utf-8-sig"
)

REFERENCE_VALUES_JSON_PATH.write_text(
    json.dumps(
        reference_records,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

REFERENCE_SCHEMA_PATH.write_text(
    json.dumps(
        REFERENCE_SCHEMA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

EXTRACTION_SCHEMA_PATH.write_text(
    json.dumps(
        EXTRACTION_SCHEMA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

EXTRACTION_TASK_PATH.write_text(
    EXTRACTION_TASK.strip(),
    encoding="utf-8",
    newline="\n"
)

REFERENCE_SUMMARY_PATH.write_text(
    json.dumps(
        REFERENCE_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

REFERENCE_METADATA_PATH.write_text(
    json.dumps(
        REFERENCE_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

DOCUMENT_CHARACTERISATION_PATH.write_text(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

QUALITY_EVIDENCE_PATH.write_text(
    json.dumps(
        QUALITY_EVIDENCE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

REFERENCE_INTEGRITY_PATH.write_text(
    json.dumps(
        REFERENCE_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

TEXT_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TEXT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

indicator_assessment_df[
    [
        "Dimension",
        "Indicator",
        "Score",
        "Evidence Source",
        "Justification"
    ]
].to_csv(
    INDICATOR_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)

dimension_assessment_df.to_csv(
    DIMENSION_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "D8 Stage 1 outputs exported."
)

D8 Stage 1 outputs exported.


In [21]:
# ============================================================
# 20. Final checks and output listing
# ============================================================

GENERATED_OUTPUTS = [
    REFERENCE_VALUES_PATH,
    REFERENCE_VALUES_JSON_PATH,
    EXTRACTION_SCHEMA_PATH,
    EXTRACTION_TASK_PATH,
    REFERENCE_SCHEMA_PATH,
    REFERENCE_SUMMARY_PATH,
    REFERENCE_METADATA_PATH,
    DOCUMENT_CHARACTERISATION_PATH,
    TEXT_DIAGNOSTICS_PATH,
    INDICATOR_ASSESSMENT_PATH,
    DIMENSION_ASSESSMENT_PATH,
    QUALITY_EVIDENCE_PATH,
    REFERENCE_INTEGRITY_PATH
]

missing_outputs = [
    path.name
    for path in GENERATED_OUTPUTS
    if not path.exists()
]

if missing_outputs:
    raise AssertionError(
        f"Missing D8 Stage 1 outputs: {missing_outputs}"
    )

if not REFERENCE_INTEGRITY[
    "reference_integrity_passed"
]:
    raise AssertionError(
        "D8 reference-integrity checks failed."
    )


print(
    "D8 Stage 1 completed successfully."
)

print()

print("Generated files:")

for path in GENERATED_OUTPUTS:
    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )


D8 Stage 1 completed successfully.

Generated files:
- D8_reference_values.csv | exists: True
- D8_reference_values.json | exists: True
- D8_extraction_schema.json | exists: True
- D8_extraction_task.txt | exists: True
- D8_reference_schema.json | exists: True
- D8_reference_summary.json | exists: True
- D8_reference_metadata.json | exists: True
- D8_document_characterisation.json | exists: True
- D8_text_diagnostics.json | exists: True
- D8_indicator_assessment.csv | exists: True
- D8_dimension_assessment.csv | exists: True
- D8_quality_evidence.json | exists: True
- D8_reference_integrity.json | exists: True
